# FAISS index builder
Builds the cosine-similarity index over the real-data chunk embeddings.

In [1]:
from pathlib import Path
import sys

# Works on every machine: find the project root (the folder that contains rag/pipeline.py)
PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "rag" / "pipeline.py").exists()
)
RAG_DIR = PROJECT_ROOT / "rag"
import numpy as np
import faiss


# ============================================================
# Configuration
# ============================================================

EMBEDDINGS_FILE = RAG_DIR / "embeddings" / "processed" / "embeddings" / "embeddings.npy"

# The dense retriever reads rag/vector_store/processed/vector_store/faiss.index
OUTPUT_DIR = RAG_DIR / "vector_store" / "processed" / "vector_store"

INDEX_FILE = OUTPUT_DIR / "faiss.index"


# ============================================================
# Load embeddings
# ============================================================

def load_embeddings():

    print("Loading embeddings...")

    embeddings = np.load(
        EMBEDDINGS_FILE
    )

    print(
        f"Loaded embeddings: {embeddings.shape}"
    )

    return embeddings


# ============================================================
# Build FAISS index
# ============================================================

def build_index(embeddings):

    dimension = embeddings.shape[1]

    print("\nBuilding FAISS index...")
    print(f"Dimension: {dimension}")

    # Inner Product + normalized vectors
    # = cosine similarity
    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(
        embeddings.astype(np.float32)
    )

    print(
        f"Vectors added: {index.ntotal}"
    )

    return index


# ============================================================
# Save index
# ============================================================

def save_index(index):

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    faiss.write_index(
        index,
        str(INDEX_FILE)
    )

    print(
        f"\nFAISS index saved to:\n"
        f"{INDEX_FILE.resolve()}"
    )


# ============================================================
# Validate index
# ============================================================

def validate_index(index, embeddings):

    print("\n================================")
    print("FAISS VALIDATION")
    print("================================")

    print(
        f"Index dimension: "
        f"{index.d}"
    )

    print(
        f"Number of vectors: "
        f"{index.ntotal}"
    )

    print(
        f"Expected vectors: "
        f"{len(embeddings)}"
    )

    if index.ntotal != len(embeddings):

        raise ValueError(
            "FAISS vector count does not "
            "match embedding count."
        )

    if index.d != embeddings.shape[1]:

        raise ValueError(
            "FAISS dimension does not "
            "match embedding dimension."
        )

    print("Vector count validation: PASSED")
    print("Dimension validation: PASSED")


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    print("================================")
    print("FLAME IN FREEFALL FAISS")
    print("================================")

    # Load embeddings
    embeddings = load_embeddings()

    # Build index
    index = build_index(
        embeddings
    )

    # Validate
    validate_index(
        index,
        embeddings
    )

    # Save
    save_index(
        index
    )

    print("\n================================")
    print("FAISS BUILD COMPLETE")
    print("================================")

FLAME IN FREEFALL FAISS
Loading embeddings...
Loaded embeddings: (771, 384)

Building FAISS index...
Dimension: 384
Vectors added: 771

FAISS VALIDATION
Index dimension: 384
Number of vectors: 771
Expected vectors: 771
Vector count validation: PASSED
Dimension validation: PASSED

FAISS index saved to:
C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\vector_store\processed\vector_store\faiss.index

FAISS BUILD COMPLETE
